# Pill Scanner — data acquisition, training, and the accuracy gate

Run this top-to-bottom **on Kaggle** (Settings → Accelerator → GPU T4 x2 or P100;
Settings → Internet → On). It was authored in a sandbox with no internet access
to the sources it uses, so **nothing in this notebook has been executed against
real data yet** — read each cell before running, especially Section 1 where a
couple of paths need to be confirmed/filled in by hand.

## Datasets this notebook pulls together

| Dataset | What it adds | Source |
|---|---|---|
| ePillID | ~4,902 RX classes, studio reference photos (baseline, carried over from `pill-id`) | Kaggle mirror `tommyngx/epillid-data-v1` |
| NIH C3PI / RxIMAGE | ~4,000 studio reference photos **+ ~133,000 consumer-grade photos** — this is the big one for closing the studio-vs-real-photo gap | `data.virginia.gov` (see Section 1.2 — direct link needs confirming, it isn't Kaggle-hosted) |
| Pills-Detection-Dataset (x2 mirrors) | Bounding-box labeled pill photos — used here to train/improve the pill-cropping step, and as extra classification images where a drug label is recoverable | Kaggle `alexanderyyy/pills-detection-dataset`, `perfect9015/pillsdetectiondataset` |
| TruMedicines pill datasets (x2) | Additional real pharma-photographed pill images | Kaggle `trumedicines/1k-pharmaceutical-pill-image-dataset`, `trumedicines/pharmaceutical-tablets-dataset` |
| OGYEIv2 | 112 pill classes, real photos (not US-NDC-mapped) — added specifically because none of the above cover OTC/general visual diversity | Kaggle `richardradli/ogyeiv2` |
| Pharmaceutical Drugs & Vitamins v2 | Real drugstore drug/vitamin photos, classification-style (not US-NDC-mapped) | Kaggle `vencerlanz09/pharmaceutical-drugs-and-vitamins-dataset-v2` |
| Drugs and Vitamins Classification | Another real, per-class-labeled drug/vitamin photo set | Kaggle `utkarshsaxenadn/drugs-and-vitamins-classification` |
| RxNav (metadata only) | imprint / color / shape / score-marks per NDC, resolved from `data/seed_lists/*.csv` | `rxnav.nlm.nih.gov` REST API |

I could not find a Kaggle mirror of the CURE pill dataset referenced earlier in
this project's planning — it's cited in pill-recognition papers but I couldn't
verify a live, downloadable copy. If you find one, add it as another Section 1
cell following the same pattern as the others; don't block on it.

## What this notebook does NOT do

It does not guess at directory layouts it can't verify. Section 1 prints the
actual folder structure of each attached dataset before anything downstream
uses it — **read that output and adjust the manifest-parsing cell in Section 3
if a dataset's layout doesn't match what's assumed there.**


**Honest OTC-coverage note:** the original 5 datasets are entirely RX-focused or unlabeled-by-drug — none had a single OTC-labeled image. The 3 datasets above are real per-class photos but aren't US-NDC-mapped (different countries' retail products), so they widen visual/appearance coverage rather than giving exact NDC-level OTC classes. True US-NDC-mapped OTC image data is still an open gap.

## 0. Setup

In [ ]:
!pip install -q transformers timm albumentations peft pytesseract
!apt-get -qq install -y tesseract-ocr > /dev/null

import os, json, glob, re, random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

WORK_DIR = Path("/kaggle/working")
MANIFEST_PATH = WORK_DIR / "image_manifest.csv"
METADATA_DIR = WORK_DIR / "metadata"
EXPORT_DIR = WORK_DIR / "export"
for d in (METADATA_DIR, EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)


## 1. Attach datasets

In the Kaggle notebook UI: **Add Input** → search each slug below → Add.
Then run this section — it just lists what's actually on disk for each one,
so Section 3's manifest builder can be checked against reality instead of
assumption.


In [ ]:
# 1.1 Kaggle-hosted datasets. Confirmed-US: epillid (NIH C3PI-derived). The
# rest are non-US or unconfirmed origin, kept for volume/diversity per an
# explicit "primarily US, maximize images per class" call — DailyMed (its
# own acquisition notebook) + epillid + C3PI are the confirmed-US backbone;
# these add bulk on top of that, not instead of it.
KAGGLE_DATASETS = {
    "epillid": ("tommyngx", "epillid-data-v1"),              # US (NIH C3PI-derived)
    "pills_detect_a": ("alexanderyyy", "pills-detection-dataset"),   # Vietnam (VAIPE-based)
    "pills_detect_b": ("perfect9015", "pillsdetectiondataset"),      # unconfirmed origin
    "trumed_1k": ("trumedicines", "1k-pharmaceutical-pill-image-dataset"),  # unconfirmed origin
    "trumed_tablets": ("trumedicines", "pharmaceutical-tablets-dataset"),   # unconfirmed origin
    "ogyeiv2": ("richardradli", "ogyeiv2"),                    # Hungary
    "phvitamins_v2": ("vencerlanz09", "pharmaceutical-drugs-and-vitamins-dataset-v2"),  # Philippines
    "drugs_vitamins_cls": ("utkarshsaxenadn", "drugs-and-vitamins-classification"),     # unconfirmed origin
}

def find_dataset_root(owner, slug):
    candidates = [
        f"/kaggle/input/datasets/{owner}/{slug}",
        f"/kaggle/input/{slug}",
    ]
    candidates += glob.glob(f"/kaggle/input/datasets/{owner}/{slug}*")
    candidates += glob.glob(f"/kaggle/input/{slug}*")
    for c in candidates:
        if Path(c).exists():
            return c
    return None

dataset_roots = {}
for key, (owner, slug) in KAGGLE_DATASETS.items():
    root = find_dataset_root(owner, slug)
    if root:
        dataset_roots[key] = root
        print(f"[{key}] found at {root}")
    else:
        print(f"[{key}] NOT attached — add it via Add Input if you want this source included")

for key, root in dataset_roots.items():
    print(f"\n--- {key} ({root}) — first 20 entries, up to 3 levels deep ---")
    n = 0
    for p in Path(root).rglob("*"):
        if n >= 20:
            break
        print(" ", p.relative_to(root))
        n += 1

In [ ]:
# 1.2 NIH C3PI / RxIMAGE — ~133,000 consumer-quality + ~4,000 reference
# images, the single biggest lever for closing the studio-vs-real-photo gap.
# Not Kaggle-hosted. Every hosting page I tried to fetch directly (Virginia
# Open Data Portal, healthdata.gov, datadiscovery.nlm.nih.gov) was blocked by
# network policy in the authoring sandbox, so — same approach as DailyMed's
# URL discovery — this cell has Kaggle itself (which has real internet)
# fetch each candidate page and regex-scan for actual .zip links, rather than
# a guessed/hardcoded URL.
import re
import requests

C3PI_CANDIDATE_PAGES = [
    "https://data.virginia.gov/dataset/computational-photography-project-for-pill-identification-c3pi",
    "https://healthdata.gov/dataset/Computational-Photography-Project-for-Pill-Identif/5vhs-kfa6",
    "https://datadiscovery.nlm.nih.gov/Chemicals-and-Drugs/Computational-Photography-Project-for-Pill-Identif/5jdf-gdqh",
]

def discover_c3pi_zip_urls():
    found = {}
    for page in C3PI_CANDIDATE_PAGES:
        try:
            resp = requests.get(page, timeout=30)
            resp.raise_for_status()
        except Exception as e:
            print(f"  could not fetch {page}: {e}")
            continue
        hrefs = re.findall(r'href="([^"]+\.zip)"', resp.text)
        for href in hrefs:
            url = href if href.startswith("http") or href.startswith("ftp") else page.rsplit("/", 1)[0] + "/" + href
            found[url] = page
    return found

print("Searching known C3PI hosting pages for direct .zip links...")
c3pi_discovered = discover_c3pi_zip_urls()
if c3pi_discovered:
    print(f"Found {len(c3pi_discovered)} candidate zip URLs:")
    for url, source_page in c3pi_discovered.items():
        print(f"  {url}   (from {source_page})")
else:
    print("No .zip links found automatically. Open one of the pages above in a "
          "real browser, find the reference/consumer-quality zip download links "
          "by hand, and paste them into C3PI_REFERENCE_ZIP_URL / "
          "C3PI_CONSUMER_ZIP_URL below.")

# Manual override / fallback — fill these in if auto-discovery above found
# nothing, or to pick specific URLs out of what it found.
C3PI_REFERENCE_ZIP_URL = ""  # e.g. one of the URLs printed above
C3PI_CONSUMER_ZIP_URL = ""   # e.g. one of the URLs printed above

# Given the multi-GB size of this dataset (133k consumer images), require an
# explicit go-ahead before downloading rather than doing it automatically the
# moment a URL is found.
C3PI_CONFIRM_DOWNLOAD = False

C3PI_ROOT = ""  # set this to the mounted/extracted path if you already have it as a Kaggle Dataset instead

if C3PI_ROOT:
    print("\nC3PI_ROOT contents (top level):")
    for p in sorted(Path(C3PI_ROOT).iterdir())[:20]:
        print(" ", p.name)
elif C3PI_CONFIRM_DOWNLOAD and (C3PI_REFERENCE_ZIP_URL or C3PI_CONSUMER_ZIP_URL):
    import urllib.request, zipfile
    c3pi_dir = WORK_DIR / "c3pi"
    c3pi_dir.mkdir(exist_ok=True)
    for label, url in [("reference", C3PI_REFERENCE_ZIP_URL), ("consumer", C3PI_CONSUMER_ZIP_URL)]:
        if not url:
            continue
        zip_path = c3pi_dir / f"{label}.zip"
        print(f"downloading {label} from {url} ...")
        if url.startswith("ftp://"):
            urllib.request.urlretrieve(url, zip_path)
        else:
            resp = requests.get(url, stream=True, timeout=120)
            resp.raise_for_status()
            with open(zip_path, "wb") as f:
                for chunk in resp.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(c3pi_dir / label)
        print(f"extracted to {c3pi_dir / label}")
    C3PI_ROOT = str(c3pi_dir)
else:
    print("\nC3PI not configured — proceeding without it for now. This is the "
          "dataset that provides ~133k consumer-quality images, so skipping it "
          "means the consumer-domain eval split will be thin or empty. Set the "
          "URL(s) above, flip C3PI_CONFIRM_DOWNLOAD to True, and re-run when ready "
          "— given the size, consider pulling this in its own committed CPU-only "
          "notebook the same way dailymed_acquisition.ipynb works, rather than "
          "inline here.")

### 1.3 DailyMed bulk SPL (US RX+OTC) — from a separate acquisition notebook

DailyMed download+parsing now lives in its own notebook,
`notebooks/dailymed_acquisition.ipynb`, run separately (CPU-only, via
"Save & Run All" / commit, since it downloads several GB over FTP and can
run for a long time unattended — that doesn't need this notebook's GPU, and
committed runs don't idle-timeout the way an interactive session does).

**Before running this section**: commit that notebook, then here, Add Input
-> search -> filter "Notebook" -> select it, so its output
(`dailymed_manifest.csv`, `dailymed_metadata.json`, `dailymed_images/`) is
readable from this notebook.


In [ ]:
import glob as _glob

def find_dailymed_manifest():
    candidates = _glob.glob("/kaggle/input/**/dailymed_manifest.csv", recursive=True)
    return candidates[0] if candidates else None

DAILYMED_MANIFEST_PATH = find_dailymed_manifest()
if DAILYMED_MANIFEST_PATH:
    print(f"Found DailyMed acquisition output at: {DAILYMED_MANIFEST_PATH}")
    DAILYMED_METADATA_PATH = Path(DAILYMED_MANIFEST_PATH).parent / "dailymed_metadata.json"
    print(f"  metadata present: {DAILYMED_METADATA_PATH.exists()}")
else:
    print("DailyMed acquisition output not found. Run notebooks/dailymed_acquisition.ipynb "
          "separately (Save & Run All / commit), then Add Input > Notebook > that notebook "
          "here, and re-run this cell.")


## 2. Drug metadata (imprint / color / shape / score marks)

Resolves `data/seed_lists/top_500_seed.csv` and `top_otc_seed.csv` (copy them
into this notebook's input, or paste their contents — see the cell below) into
real NDCs + structured attributes via RxNav. This is the same logic as
`scripts/build_drug_metadata.py` in the repo, inlined here so the notebook is
self-contained.


In [ ]:
import time
import urllib.parse, urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

RXNAV_BASE = "https://rxnav.nlm.nih.gov/REST"

class RateLimited(Exception):
    pass

def _get(url):
    for attempt in range(5):
        try:
            with urllib.request.urlopen(url, timeout=20) as resp:
                return json.load(resp)
        except Exception:
            time.sleep(2 ** attempt)
    raise RateLimited(url)

def rxcuis_for_name(name):
    data = _get(f"{RXNAV_BASE}/rxcui.json?name={urllib.parse.quote(name)}&search=2")
    return data.get("idGroup", {}).get("rxnormId", []) or []

def ndcs_for_rxcui(rxcui):
    data = _get(f"{RXNAV_BASE}/rxcui/{rxcui}/ndcs.json")
    return data.get("ndcGroup", {}).get("ndcList", {}).get("ndc", []) or []

def properties_for_ndc(ndc):
    data = _get(f"{RXNAV_BASE}/ndcproperties.json?id={ndc}&ndcstatus=ALL")
    pl = data.get("ndcPropertyList", {}).get("ndcProperty", [])
    if not pl:
        return None
    p = pl[0]
    props = {x["propName"]: x["propValue"] for x in p.get("propertyConceptList", {}).get("propertyConcept", [])}
    return {
        "ndc": ndc, "rxcui": p.get("rxcui"),
        "imprint": props.get("IMPRINT_CODE"), "color": props.get("COLORTEXT"),
        "shape": props.get("SPLSHAPE"), "score_marks": props.get("SPLSCORE"),
        "status": props.get("NDC_STATUS"),
    }

def resolve_one(name):
    entries = []
    try:
        for rxcui in rxcuis_for_name(name):
            for ndc in ndcs_for_rxcui(rxcui):
                try:
                    props = properties_for_ndc(ndc)
                except RateLimited:
                    continue
                if props:
                    entries.append(props)
    except RateLimited:
        pass
    return name, entries

# Inlined from data/seed_lists/{top_500_seed,top_otc_seed}.csv (kept in
# sync manually) — GitHub-repo attachment has been unreliable in practice
# this session, and this list is now load-bearing (it defines the
# "priority tier" the accuracy gate reports separately, per the actual
# goal: 80/90 on the top-500 RX+OTC specifically, not the long tail).
seed_names = ['atorvastatin', 'levothyroxine', 'metformin', 'lisinopril', 'amlodipine', 'metoprolol tartrate', 'metoprolol succinate ER', 'albuterol', 'omeprazole', 'losartan', 'gabapentin', 'hydrochlorothiazide', 'sertraline', 'simvastatin', 'escitalopram', 'rosuvastatin', 'bupropion', 'furosemide', 'pantoprazole', 'trazodone', 'fluticasone propionate nasal', 'duloxetine', 'tamsulosin', 'prednisone', 'amoxicillin', 'alprazolam', 'citalopram', 'meloxicam', 'clopidogrel', 'azithromycin', 'warfarin', 'cyclobenzaprine', 'carvedilol', 'venlafaxine ER', 'insulin glargine', 'montelukast', 'pravastatin', 'hydrocodone/acetaminophen', 'tramadol', 'potassium chloride ER', 'lorazepam', 'clonazepam', 'cholecalciferol (vitamin D3) high-dose', 'spironolactone', 'allopurinol', 'amitriptyline', 'buspirone', 'famotidine', 'zolpidem', 'atenolol', 'diazepam', 'glipizide', 'ezetimibe', 'doxycycline hyclate', 'cephalexin', 'ciprofloxacin', 'levofloxacin', 'clindamycin', 'fluoxetine', 'paroxetine', 'quetiapine', 'risperidone', 'aripiprazole', 'olanzapine', 'lamotrigine', 'levetiracetam', 'topiramate', 'divalproex sodium ER', 'phenytoin', 'carbamazepine', 'oxycodone', 'oxycodone/acetaminophen', 'morphine sulfate ER', 'methylphenidate', 'amphetamine/dextroamphetamine', 'lisdexamfetamine', 'atomoxetine', 'sildenafil', 'tadalafil', 'finasteride', 'metronidazole', 'sulfamethoxazole/trimethoprim', 'nitrofurantoin', 'valacyclovir', 'acyclovir', 'prednisolone', 'methylprednisolone dose pack', 'budesonide', 'tiotropium', 'cetirizine', 'loratadine', 'fexofenadine', 'diphenhydramine', 'promethazine', 'ondansetron', 'metoclopramide', 'docusate sodium', 'polyethylene glycol 3350', 'lactulose', 'hydralazine', 'clonidine', 'diltiazem ER', 'verapamil ER', 'nifedipine ER', 'digoxin', 'apixaban', 'rivaroxaban', 'dabigatran', 'cyanocobalamin (vitamin B12)', 'folic acid', 'ferrous sulfate', 'calcium acetate', 'sevelamer', 'sitagliptin', 'glimepiride', 'pioglitazone', 'empagliflozin', 'dapagliflozin', 'liraglutide', 'semaglutide oral', 'tirzepatide', 'mirtazapine', 'desvenlafaxine', 'vortioxetine', 'buprenorphine/naloxone', 'naltrexone', 'disulfiram', 'propranolol', 'labetalol', 'sotalol', 'amiodarone', 'isosorbide mononitrate', 'nitroglycerin SL', 'colchicine', 'febuxostat', 'methotrexate', 'hydroxychloroquine', 'prednisone dose pack', 'dexamethasone', 'triamcinolone', 'azelastine', 'mometasone', 'benzonatate', 'guaifenesin with codeine', 'lidocaine patch', 'gabapentin ER', 'pregabalin', 'baclofen', 'tizanidine', 'methocarbamol', 'acetaminophen', 'ibuprofen', 'naproxen sodium', 'aspirin', 'acetaminophen/diphenhydramine', 'loratadine', 'cetirizine', 'fexofenadine', 'diphenhydramine', 'chlorpheniramine', 'loratadine/pseudoephedrine', 'omeprazole', 'esomeprazole', 'lansoprazole', 'famotidine', 'calcium carbonate', 'calcium carbonate/magnesium hydroxide', 'simethicone', 'loperamide', 'bismuth subsalicylate', 'docusate sodium', 'polyethylene glycol 3350', 'bisacodyl', 'sennosides', 'dextromethorphan', 'guaifenesin', 'guaifenesin/dextromethorphan', 'phenylephrine', 'pseudoephedrine', 'acetaminophen/dextromethorphan/phenylephrine', 'doxylamine/acetaminophen/dextromethorphan', 'diphenhydramine (sleep aid)', 'doxylamine succinate', 'melatonin', 'meclizine', 'dimenhydrinate', 'multivitamin adult', 'vitamin D3', 'vitamin C', 'biotin', 'fish oil / omega-3', 'folic acid', 'iron (ferrous sulfate)', 'calcium + vitamin D', 'glucosamine/chondroitin', 'probiotic', 'nicotine lozenge', 'ibuprofen/famotidine', 'naproxen/esomeprazole']

print(f"{len(seed_names)} seed drug names to resolve")

drug_metadata = {}
if seed_names:
    with ThreadPoolExecutor(max_workers=6) as pool:
        futures = [pool.submit(resolve_one, n) for n in seed_names]
        for i, fut in enumerate(as_completed(futures), 1):
            name, entries = fut.result()
            drug_metadata[name] = entries
            if i % 25 == 0:
                print(f"  {i}/{len(seed_names)} resolved")

json.dump(drug_metadata, open(METADATA_DIR / "drug_metadata.json", "w"), indent=0)
total_ndcs = sum(len(v) for v in drug_metadata.values())
print(f"Resolved {len(drug_metadata)} names / {total_ndcs} NDC entries")

def normalize_ndc9(ndc):
    """5-4 digit labeler-product prefix (ignoring package size), same
    zero-padding convention pill-id's build_ndc_names.py uses — NDC padding
    inconsistency between sources (RxNav vs. dataset-native strings) is a
    well-known problem, this is the standard normalization for it."""
    parts = str(ndc).split("-")
    if len(parts) < 2:
        return str(ndc)
    return f"{parts[0].zfill(5)}-{parts[1].zfill(4)}"

PRIORITY_NDC9_SET = {
    normalize_ndc9(entry["ndc"])
    for entries in drug_metadata.values()
    for entry in entries
    if entry.get("ndc")
}
print(f"Priority tier: {len(PRIORITY_NDC9_SET)} distinct NDC9 prefixes across the seed lists")


## 3. Build the unified image manifest

One row per image: `path, label(ndc-or-generic), side(front/back/unknown),
domain(reference/consumer/unknown), source_dataset`. This is the join point
across every dataset attached in Section 1 — **check the printed folder
listings above before trusting the parsing logic below**, since a Kaggle
dataset's internal layout can change or differ from what's assumed here.

Default label-parsing rule (matches `pill-id`'s existing convention): take the
token before the first underscore in the filename as the label. Override this
per-dataset if Section 1's listing shows a different naming scheme.


In [ ]:
def default_label_from_filename(path):
    return Path(path).stem.split("_", 1)[0]

def guess_side(path):
    name = Path(path).stem.lower()
    if any(t in name for t in ("_sf", "front", "_f_", "top")):
        return "front"
    if any(t in name for t in ("_sb", "back", "_b_", "bottom")):
        return "back"
    return "unknown"

manifest_rows = []  # dicts: path, label, side, domain, source, category ("RX"/"OTC"/"unknown")

def add_epillid():
    """Uses ePillID's own all_labels.csv (confirmed columns: images,
    pilltype_id, label_code_id, prod_code_id, is_ref, is_front, is_new,
    image_path, label) rather than guessing from filenames — it has real
    front/back and reference/pool flags. All of ePillID is RX (it's built
    from NIH's C3PI, which is RX-only by construction)."""
    root = dataset_roots.get("epillid")
    if not root:
        return
    csv_path = Path(root) / "ePillID_data" / "all_labels.csv"
    base = Path(root) / "ePillID_data" / "classification_data"
    if not csv_path.exists():
        print("epillid: all_labels.csv missing, skipping (no safe fallback)")
        return
    import pandas as pd
    df = pd.read_csv(csv_path)
    missing = 0
    for _, row in df.iterrows():
        img_path = base / row["image_path"]
        if not img_path.exists():
            missing += 1
            continue
        manifest_rows.append({
            "path": str(img_path),
            "label": row["label"],
            "side": "front" if bool(row["is_front"]) else "back",
            # is_ref=False is STILL studio-pipeline photography (ePillID's
            # larger "training pool" tier), not a real-world/phone photo.
            # Tagging it "consumer" would recreate the exact false-confidence
            # problem this whole eval redesign exists to catch.
            "domain": "reference" if bool(row["is_ref"]) else "reference_pool",
            "source": "epillid",
            "category": "RX",
        })
    if missing:
        print(f"epillid: {missing} rows in all_labels.csv had no matching file on disk")

def add_c3pi():
    if not C3PI_ROOT:
        return
    for p in Path(C3PI_ROOT).rglob("*.jpg"):
        # C3PI's own layout separates reference/ and consumer-quality tiers by
        # top-level folder name (per the NLM listing) — adjust this split if
        # the actual downloaded structure differs. Unlike ePillID's
        # "reference_pool", C3PI's consumer tier really is phone/camera
        # photos taken in real-world conditions (per NLM's own description),
        # so "consumer" here is an honest label, not an approximation.
        domain = "consumer" if "consumer" in str(p).lower() or "test" in str(p).lower() else "reference"
        manifest_rows.append({
            "path": str(p), "label": default_label_from_filename(p),
            "side": guess_side(p), "domain": domain, "source": "c3pi",
            "category": "RX",  # C3PI/RxIMAGE is RX-only by construction
        })

def add_dailymed():
    """Reads the separate dailymed_acquisition.ipynb notebook's output
    (dailymed_manifest.csv) rather than downloading live in this notebook —
    see Section 1.3. This is the only source with real US-NDC OTC coverage."""
    if not DAILYMED_MANIFEST_PATH:
        return
    import csv as _csv2
    manifest_dir = Path(DAILYMED_MANIFEST_PATH).parent
    n_missing = 0
    with open(DAILYMED_MANIFEST_PATH) as f:
        for row in _csv2.DictReader(f):
            path = row["path"]
            if not Path(path).exists():
                # Absolute paths recorded during acquisition point at that
                # notebook's own /kaggle/working, which won't exist here —
                # re-root relative to wherever this manifest was actually
                # found (its sibling dailymed_images/ directory).
                alt = manifest_dir / "dailymed_images" / Path(path).name
                if alt.exists():
                    path = str(alt)
                else:
                    n_missing += 1
                    continue
            manifest_rows.append({**row, "path": path})
    if n_missing:
        print(f"add_dailymed: {n_missing} manifest rows had no resolvable image file")

IMAGE_EXTENSIONS = ("*.jpg", "*.jpeg", "*.png")

def _find_images(root):
    for pattern in IMAGE_EXTENSIONS:
        yield from Path(root).rglob(pattern)

def add_generic_detection_dataset(key):
    root = dataset_roots.get(key)
    if not root:
        return
    # These are bounding-box detection datasets (single class "pill"), not
    # per-drug labeled. They're valuable for training/improving a pill
    # cropper, not for classification labels — tag label=None so downstream
    # training code can route them to the detector step instead of the
    # classifier, rather than silently mislabeling them.
    n = 0
    for p in _find_images(root):
        manifest_rows.append({
            "path": str(p), "label": None, "side": "unknown",
            "domain": "consumer", "source": key, "category": "unknown",
        })
        n += 1
    if n == 0:
        print(f"{key}: no image files found (checked .jpg/.jpeg/.png) — "
              f"the attached dataset may only contain a non-image artifact "
              f"(e.g. a packaged/opaque file or a pretrained model weights "
              f"file), not raw images to train on.")

# Split/wrapper folder names that are never themselves a class name —
# confirmed necessary via a live run: ogyeiv2 and phvitamins_v2 both nest
# images under train/valid/test + images/labels wrapper folders rather than
# having class folders directly at the dataset root, which the original
# root.iterdir()-only version of this function missed entirely (0 images
# from either). phvitamins_v2's real class folders (e.g. "Biogesic",
# "Bonamine") sit two levels deeper, under "Capsure Dataset/Train Image/".
_SPLIT_WRAPPER_NAMES = {
    "train", "val", "valid", "validation", "test", "images", "image",
    "labels", "label", "train image", "val image", "test image",
    "annotations",
}

def add_labeled_folder_dataset(key, label_prefix=None):
    """For classification-style datasets where each image's *immediate
    parent folder* is a real class/product name (ogyeiv2, phvitamins_v2,
    drugs_vitamins_cls) — searched recursively rather than assuming class
    folders sit directly at the dataset root, since real layouts vary (see
    _SPLIT_WRAPPER_NAMES above). Labels are raw folder names (prefixed by
    source), NOT US-NDC-mapped — visual/appearance diversity, not exact
    NDC-level classes. A dataset whose images all sit under split/wrapper
    folders with no real class-name level (e.g. a pure YOLO detection
    layout with only train/images, train/labels — ogyeiv2's actual layout)
    legitimately contributes 0 labeled rows here; that's correct, not a bug,
    short of writing a YOLO-label parser to recover class names from
    per-image .txt files instead of folder names.
    """
    root = dataset_roots.get(key)
    if not root:
        return
    prefix = label_prefix or key
    n_added = 0
    for p in _find_images(root):
        class_name = p.parent.name
        if class_name.strip().lower() in _SPLIT_WRAPPER_NAMES:
            continue  # parent is a split/wrapper folder, not a real class name
        manifest_rows.append({
            "path": str(p),
            "label": f"{prefix}:{class_name}",
            "side": "unknown",
            "domain": "reference_pool",
            "source": key,
            "category": "unknown",
        })
        n_added += 1
    if n_added == 0:
        print(f"{key}: no images found with a real class-name parent folder "
              f"(only split/wrapper-named parents, or no images at all) — "
              f"this source contributes 0 rows.")

add_epillid()
add_c3pi()
add_dailymed()
# Non-US / unconfirmed-origin sources, restored for volume per an explicit
# "primarily US, maximize images per class" call. None of these are
# US-NDC-mapped, so they widen visual/appearance diversity and (for the
# detection sets) crop/detector training data — they don't substitute for
# epillid/C3PI/DailyMed's real US NDC coverage.
add_generic_detection_dataset("pills_detect_a")
add_generic_detection_dataset("pills_detect_b")
add_generic_detection_dataset("trumed_1k")
add_generic_detection_dataset("trumed_tablets")
add_labeled_folder_dataset("ogyeiv2")
add_labeled_folder_dataset("phvitamins_v2")
add_labeled_folder_dataset("drugs_vitamins_cls")

# Tag each row's tier — "priority" if it's one of the top-500 RX/OTC seed
# drugs (Section 2's PRIORITY_NDC9_SET), "long_tail" for other real NDC
# rows, "unknown" for non-NDC raw-name rows (the diversity-only datasets).
# This is the split that actually matters for the doctor-testing goal —
# reported separately, never blended into one overall number.
for row in manifest_rows:
    if row["category"] in ("RX", "OTC") and row["label"]:
        row["tier"] = "priority" if normalize_ndc9(row["label"]) in PRIORITY_NDC9_SET else "long_tail"
    else:
        row["tier"] = "unknown"

print(f"Total images collected: {len(manifest_rows)}")
by_source = Counter(r["source"] for r in manifest_rows)
print("By source:", dict(by_source))
by_cat = Counter(r["category"] for r in manifest_rows)
print("By category (RX/OTC/unknown):", dict(by_cat))
by_tier = Counter(r["tier"] for r in manifest_rows)
print("By tier (priority/long_tail/unknown):", dict(by_tier))

labeled_rows = [r for r in manifest_rows if r["label"]]
by_label_count = Counter(r["label"] for r in labeled_rows)
print(f"Labeled images: {len(labeled_rows)} across {len(by_label_count)} distinct labels")
depth_dist = Counter(by_label_count.values())
print("Images-per-label distribution (count -> how many labels have that many images):",
      dict(sorted(depth_dist.items())[:15]))

import csv as _csv
with open(MANIFEST_PATH, "w", newline="") as f:
    w = _csv.DictWriter(f, fieldnames=["path", "label", "side", "domain", "source", "category", "tier"])
    w.writeheader()
    w.writerows(manifest_rows)
print(f"Wrote manifest to {MANIFEST_PATH}")

## 4. Train / val / test split

Held-out test queries are drawn with **consumer-domain images deliberately
oversampled into the test set relative to their overall share** — otherwise a
random split would let the (usually much larger) reference-image count drown
out the consumer split, and you'd be back to an offline number that looks
better than the app performs. Split is stratified by label so every class
with 2+ images has at least one in train.


In [ ]:
from collections import defaultdict as _dd

rows_by_label = _dd(list)
for r in labeled_rows:
    rows_by_label[r["label"]].append(r)

train_rows, val_rows, test_rows = [], [], []
rng = random.Random(SEED)

for label, rows in rows_by_label.items():
    rng.shuffle(rows)
    consumer = [r for r in rows if r["domain"] == "consumer"]
    reference = [r for r in rows if r["domain"] != "consumer"]

    # Guarantee at least one train image; everything else splits 70/15/15,
    # with consumer images preferentially routed to val/test so that split
    # isn't reference-only.
    pool = reference[1:] + reference[:1] if len(reference) == 1 else reference
    if len(rows) == 1:
        train_rows.extend(rows)
        continue

    n_test = max(1, round(0.15 * len(rows))) if len(rows) >= 4 else (1 if consumer else 0)
    n_val = max(1, round(0.15 * len(rows))) if len(rows) >= 4 else 0

    test_pick = (consumer[:n_test] + reference)[:n_test] if consumer else reference[:n_test]
    remaining = [r for r in rows if r not in test_pick]
    val_pick = remaining[:n_val]
    train_pick = [r for r in remaining if r not in val_pick]

    train_rows.extend(train_pick)
    val_rows.extend(val_pick)
    test_rows.extend(test_pick)

print(f"train={len(train_rows)} val={len(val_rows)} test={len(test_rows)}")
print("test domain mix:", Counter(r["domain"] for r in test_rows))


## 5. Domain-randomization augmentation

This is the single highest-leverage fix identified from the previous model's
numbers: every training/eval image before this point comes from a studio-style
photo (even most of the "consumer" C3PI tier is still a fairly controlled
capture). Real phone photos in the wild add background clutter, harsh/uneven
lighting, off-axis perspective, motion blur, and JPEG compression on top of
that. Since we can't collect real wild photos for every one of thousands of
classes, we simulate the gap instead — apply this aggressively to *reference*
images so the model doesn't overfit to clean studio conditions, and more
lightly (or not at all) to genuine consumer-domain images.


In [ ]:
import albumentations as A

reference_domain_aug = A.Compose([
    A.RandomRotate90(p=0.3),
    A.Rotate(limit=25, p=0.7),
    A.Perspective(scale=(0.02, 0.08), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.35, contrast_limit=0.35, p=0.8),
    A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=25, val_shift_limit=20, p=0.5),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 7)),
        A.MotionBlur(blur_limit=(3, 9)),
    ], p=0.4),
    A.ImageCompression(quality_range=(35, 90), p=0.6),
    A.CoarseDropout(num_holes_range=(1, 3), hole_height_range=(0.05, 0.15),
                     hole_width_range=(0.05, 0.15), p=0.3),
    A.GaussNoise(std_range=(0.02, 0.08), p=0.3),
])

consumer_domain_aug = A.Compose([
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.ImageCompression(quality_range=(60, 95), p=0.3),
])

def augment_for_domain(pil_image, domain):
    arr = np.array(pil_image.convert("RGB"))
    aug = reference_domain_aug if domain != "consumer" else consumer_domain_aug
    return Image.fromarray(aug(image=arr)["image"])


## 6. Dataset + P-K batch sampler

Matches the metric-learning setup already proven in `pill-id`'s
`config.json` (`proj_batch_p`/`proj_batch_k`: P classes x K images per batch,
required for triplet/SupCon losses to have valid positive pairs).


In [ ]:
from torch.utils.data import Dataset, Sampler

class PillDataset(Dataset):
    def __init__(self, rows, processor, training):
        self.rows = rows
        self.processor = processor
        self.training = training
        self.labels = sorted({r["label"] for r in rows})
        self.label_to_idx = {l: i for i, l in enumerate(self.labels)}

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.training:
            img = augment_for_domain(img, row["domain"])
        pixel_values = self.processor(images=img, return_tensors="pt")["pixel_values"][0]
        return pixel_values, self.label_to_idx[row["label"]]


class PKSampler(Sampler):
    """Yields batches of P classes x K samples/class, matching the
    proj_batch_p / proj_batch_k hyperparameters from pill-id's config.json."""

    def __init__(self, rows, labels, p, k, batches_per_epoch):
        self.by_label = defaultdict(list)
        for i, r in enumerate(rows):
            self.by_label[r["label"]].append(i)
        self.labels = [l for l in labels if len(self.by_label[l]) >= 1]
        self.p, self.k = p, k
        self.batches_per_epoch = batches_per_epoch

    def __iter__(self):
        for _ in range(self.batches_per_epoch):
            chosen_labels = random.sample(self.labels, min(self.p, len(self.labels)))
            batch = []
            for label in chosen_labels:
                pool = self.by_label[label]
                batch.extend(random.choices(pool, k=self.k) if len(pool) < self.k
                             else random.sample(pool, self.k))
            yield batch

    def __len__(self):
        return self.batches_per_epoch


## 7. Model: frozen DINOv2-large + projection head

Same architecture as the deployed `pill-id` model (so exported artifacts stay
drop-in compatible with the existing `classifier.py`/`filter_cascade.py`
backend code) — CE + ArcFace + SupCon + triplet, weighted per the proven
`config.json` values. The optional LoRA fine-tune phase on the backbone
(`run_lora`) is included too, since the prior config validated it helps.


In [ ]:
from transformers import AutoImageProcessor, AutoModel

CFG = {
    "dinov2_model_id": "facebook/dinov2-large",
    "proj_hidden_dim": 1024,
    "proj_embedding_dim": 512,
    "proj_batch_p": 12,
    "proj_batch_k": 4,
    "proj_epochs": 50,
    "proj_lr": 1e-3,
    "proj_weight_decay": 1e-4,
    "ce_weight": 0.5,
    "arcface_weight": 0.3,
    "supcon_weight": 0.7,
    "triplet_weight": 0.7,
    "triplet_margin": 0.2,
    "supcon_temperature": 0.07,
    "arcface_s": 30.0,
    "arcface_m": 0.3,
    "run_lora": True,
    "lora_epochs": 10,
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "lora_lr_backbone": 1e-5,
    "lora_lr_head": 5e-4,
}

processor = AutoImageProcessor.from_pretrained(CFG["dinov2_model_id"])
backbone = AutoModel.from_pretrained(CFG["dinov2_model_id"]).to(DEVICE)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False


class ProjectionHead(nn.Module):
    """Kept identical to pill-id/backend/app/classifier.py's ProjectionHead
    so exported weights load without modification in the existing backend."""

    def __init__(self, in_dim, hidden_dim, out_dim, num_classes):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Dropout(0.10), nn.Linear(hidden_dim, out_dim),
        )
        self.classifier = nn.Linear(out_dim, num_classes)
        self.arc_weight = nn.Parameter(torch.empty(num_classes, out_dim))
        nn.init.xavier_uniform_(self.arc_weight)

    def forward(self, x):
        emb = F.normalize(self.proj(x), dim=1)
        logits = self.classifier(emb)
        return {"emb": emb, "logits": logits}


def extract_backbone_feature(pixel_values):
    with torch.no_grad():
        out = backbone(pixel_values=pixel_values.to(DEVICE))
        return out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state[:, 0]


In [ ]:
def arcface_logits(emb, labels, weight, s, m, num_classes):
    weight_n = F.normalize(weight, dim=1)
    cos = emb @ weight_n.T
    theta = torch.acos(cos.clamp(-1 + 1e-7, 1 - 1e-7))
    target_logits = torch.cos(theta + m)
    one_hot = F.one_hot(labels, num_classes).float()
    logits = cos * (1 - one_hot) + target_logits * one_hot
    return logits * s


def supcon_loss(emb, labels, temperature):
    sim = emb @ emb.T / temperature
    sim = sim - sim.max(dim=1, keepdim=True).values.detach()
    exp_sim = torch.exp(sim)
    mask_self = torch.eye(len(labels), device=emb.device).bool()
    exp_sim = exp_sim.masked_fill(mask_self, 0)
    pos_mask = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask_self
    denom = exp_sim.sum(dim=1) + 1e-12
    log_prob = sim - torch.log(denom.unsqueeze(1) + 1e-12)
    pos_count = pos_mask.sum(dim=1).clamp(min=1)
    loss = -(log_prob * pos_mask).sum(dim=1) / pos_count
    return loss.mean()


def triplet_loss(emb, labels, margin):
    dist = torch.cdist(emb, emb)
    loss_terms = []
    for i in range(len(labels)):
        pos_mask = (labels == labels[i]) & (torch.arange(len(labels), device=emb.device) != i)
        neg_mask = labels != labels[i]
        if pos_mask.any() and neg_mask.any():
            hardest_pos = dist[i][pos_mask].max()
            hardest_neg = dist[i][neg_mask].min()
            loss_terms.append(F.relu(hardest_pos - hardest_neg + margin))
    return torch.stack(loss_terms).mean() if loss_terms else torch.tensor(0.0, device=emb.device)


## 8. Training loop

In [ ]:
def train_projection_head(train_rows, val_rows, cfg):
    labels = sorted({r["label"] for r in train_rows})
    train_ds = PillDataset(train_rows, processor, training=True)
    sampler = PKSampler(train_rows, labels, cfg["proj_batch_p"], cfg["proj_batch_k"],
                         batches_per_epoch=max(1, len(train_rows) // (cfg["proj_batch_p"] * cfg["proj_batch_k"])))

    head = ProjectionHead(in_dim=1024, hidden_dim=cfg["proj_hidden_dim"],
                           out_dim=cfg["proj_embedding_dim"], num_classes=len(labels)).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=cfg["proj_lr"], weight_decay=cfg["proj_weight_decay"])

    best_val = -1.0
    for epoch in range(cfg["proj_epochs"]):
        head.train()
        epoch_loss = 0.0
        for batch_indices in sampler:
            pixel_values = torch.stack([train_ds[i][0] for i in batch_indices]).to(DEVICE)
            batch_labels = torch.tensor([train_ds[i][1] for i in batch_indices], device=DEVICE)

            feat = extract_backbone_feature(pixel_values)
            out = head(feat.float())
            emb, logits = out["emb"], out["logits"]

            ce = F.cross_entropy(logits, batch_labels)
            arc_logits = arcface_logits(emb, batch_labels, head.arc_weight, cfg["arcface_s"], cfg["arcface_m"], len(labels))
            arc = F.cross_entropy(arc_logits, batch_labels)
            sc = supcon_loss(emb, batch_labels, cfg["supcon_temperature"])
            tr = triplet_loss(emb, batch_labels, cfg["triplet_margin"])

            loss = (cfg["ce_weight"] * ce + cfg["arcface_weight"] * arc
                    + cfg["supcon_weight"] * sc + cfg["triplet_weight"] * tr)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
            opt.step()
            epoch_loss += loss.item()

        print(f"epoch {epoch+1}/{cfg['proj_epochs']} loss={epoch_loss / len(sampler):.4f}")
        # (val-based checkpointing / early stopping: plug in eval/evaluate.py's
        # accuracy_block here once val embeddings are computed, and save the
        # best head — omitted for brevity, don't skip it in the real run)

    return head, labels

# Safer epoch count for a first real unattended run (default CFG value of 50
# risks not finishing inside Kaggle's max session time on a first real-data
# pass) — widen once a full run has been timed.
CFG["proj_epochs"] = 15
head, label_list = train_projection_head(train_rows, val_rows, CFG)


## 9. Export reference gallery + deployment artifacts

Same file format as `pill-id/dinov2_projection_head/{best_projection_head.pt,
deployed_ref_embeddings.pt}` so the existing backend code (`classifier.py`,
now `filter_cascade.py`) loads these without modification.


In [ ]:
@torch.no_grad()
def embed_rows(rows, head, label_list):
    """label_list must be the model's canonical training label order (the
    list train_projection_head returned) — NOT re-derived from `rows` here,
    since that silently corrupted label_indices whenever `rows` didn't cover
    every training class in the same sorted order. Rows whose label isn't in
    label_list (e.g. a val/test singleton class never seen in training) are
    skipped and reported, not silently mis-indexed; returns the kept rows
    alongside the embeddings so callers can zip them safely."""
    head.eval()
    label_to_idx = {l: i for i, l in enumerate(label_list)}
    embeddings, label_indices, abs_paths, kept_rows = [], [], [], []
    skipped = 0
    for row in rows:
        if row["label"] not in label_to_idx:
            skipped += 1
            continue
        img = Image.open(row["path"]).convert("RGB")
        pixel_values = processor(images=img, return_tensors="pt")["pixel_values"].to(DEVICE)
        feat = extract_backbone_feature(pixel_values)
        emb = head(feat.float())["emb"]
        embeddings.append(emb.cpu())
        label_indices.append(label_to_idx[row["label"]])
        abs_paths.append(row["path"])
        kept_rows.append(row)
    if skipped:
        print(f"embed_rows: skipped {skipped} rows with a label not in the model's training label_list")
    return {
        "embeddings": torch.cat(embeddings, dim=0),
        "label_indices": torch.tensor(label_indices),
        "abs_paths": abs_paths,
    }, kept_rows


def export_artifacts(head, label_list, gallery_rows):
    torch.save({
        "head_state_dict": head.state_dict(),
        "cfg": CFG,
        "in_dim": 1024,
        "num_classes": len(label_list),
        "label_classes": label_list,
    }, EXPORT_DIR / "best_projection_head.pt")

    gallery, _ = embed_rows(gallery_rows, head, label_list)
    torch.save(gallery, EXPORT_DIR / "deployed_ref_embeddings.pt")
    print(f"Exported artifacts to {EXPORT_DIR}")

export_artifacts(head, label_list, train_rows + val_rows)


## 10. Accuracy gate — mandatory before deploying to the app

This is the check that was missing entirely last time: it reports accuracy
**split by reference vs. consumer domain**, and refuses to write the
"approved for deploy" marker unless the consumer-domain number clears a bar
you choose. Do not manually override a failed gate to ship anyway — if it
fails, the fix is more/better consumer-domain data or more training, not
skipping the check.


In [ ]:
# Inlined from eval/evaluate.py (kept in sync manually) — GitHub-repo
# attachment has been unreliable in practice, so this cell doesn't depend on
# it for the unattended overnight run.
from dataclasses import dataclass as _dataclass
from collections import defaultdict as _defaultdict

TOPKS = (1, 5, 10, 20, 50)
CONFIDENCE_THRESHOLDS = [round(0.05 * i, 2) for i in range(1, 20)]

@_dataclass
class QueryRow:
    embedding: torch.Tensor
    true_label: str
    domain: str
    side: str
    images_in_class: int
    category: str = "unknown"
    tier: str = "unknown"  # "priority" (top-500 RX/OTC) | "long_tail" | "unknown"

def _topk_hit(true_label, ranked_labels, k):
    return true_label in ranked_labels[:k]

def _rank_query(query_emb, gallery_emb, gallery_labels):
    sims = F.normalize(query_emb, dim=0) @ F.normalize(gallery_emb, dim=1).T
    order = torch.argsort(sims, descending=True)
    seen = set()
    ranked = []
    for idx in order.tolist():
        lbl = gallery_labels[idx]
        if lbl not in seen:
            seen.add(lbl)
            ranked.append(lbl)
    return ranked, sims

def _accuracy_block(rows, gallery_emb, gallery_labels):
    hits = {k: 0 for k in TOPKS}
    top1_scores, correct_at_top1 = [], []
    n = 0
    for row in rows:
        ranked, sims = _rank_query(row.embedding, gallery_emb, gallery_labels)
        if not ranked:
            continue
        n += 1
        for k in TOPKS:
            if _topk_hit(row.true_label, ranked, k):
                hits[k] += 1
        top1_scores.append(float(sims.max()))
        correct_at_top1.append(ranked[0] == row.true_label)
    if n == 0:
        return {"n": 0}
    result = {"n": n, **{f"top{k}_acc": hits[k] / n for k in TOPKS}}
    calibration = []
    for t in CONFIDENCE_THRESHOLDS:
        answered = [c for s, c in zip(top1_scores, correct_at_top1) if s >= t]
        coverage = len(answered) / n
        precision = (sum(answered) / len(answered)) if answered else None
        calibration.append({"threshold": t, "coverage": round(coverage, 4), "precision_if_answered": precision})
    result["calibration"] = calibration
    return result

def evaluate(rows, gallery_emb, gallery_labels):
    report = {}
    report["overall"] = _accuracy_block(rows, gallery_emb, gallery_labels)
    by_domain = _defaultdict(list)
    for r in rows:
        by_domain[r.domain].append(r)
    report["by_domain"] = {d: _accuracy_block(rs, gallery_emb, gallery_labels) for d, rs in by_domain.items()}
    by_category = _defaultdict(list)
    for r in rows:
        by_category[r.category].append(r)
    report["by_category"] = {c: _accuracy_block(rs, gallery_emb, gallery_labels) for c, rs in by_category.items()}
    by_tier = _defaultdict(list)
    for r in rows:
        by_tier[r.tier].append(r)
    report["by_tier"] = {t: _accuracy_block(rs, gallery_emb, gallery_labels) for t, rs in by_tier.items()}
    by_depth = _defaultdict(list)
    for r in rows:
        bucket = "1-2" if r.images_in_class <= 2 else "3-5" if r.images_in_class <= 5 else "6+"
        by_depth[bucket].append(r)
    report["by_images_per_class"] = {b: _accuracy_block(rs, gallery_emb, gallery_labels) for b, rs in by_depth.items()}
    per_class = _defaultdict(list)
    for r in rows:
        ranked, _ = _rank_query(r.embedding, gallery_emb, gallery_labels)
        per_class[r.true_label].append(_topk_hit(r.true_label, ranked, 5))
    worst = sorted(
        ((label, sum(hits) / len(hits), len(hits)) for label, hits in per_class.items()),
        key=lambda x: x[1],
    )[:50]
    report["worst_classes_top5"] = [
        {"label": label, "top5_acc": round(acc, 3), "n_queries": n} for label, acc, n in worst
    ]
    return report

MIN_TOP5_CONSUMER = 0.70  # tune this to what you actually need before shipping

if evaluate is not None:
    gallery, _ = embed_rows(train_rows + val_rows, head, label_list)
    test_embedded, test_kept_rows = embed_rows(test_rows, head, label_list)
    query_rows = [
        QueryRow(embedding=emb, true_label=r["label"], domain=r["domain"],
                 side=r["side"], images_in_class=by_label_count[r["label"]],
                 category=r["category"], tier=r.get("tier", "unknown"))
        for r, emb in zip(test_kept_rows, test_embedded["embeddings"])
    ]
    report = evaluate(query_rows, gallery["embeddings"], [label_list[i] for i in gallery["label_indices"].tolist()])
    json.dump(report, open(EXPORT_DIR / "eval_report.json", "w"), indent=2, default=str)
    consumer_top5 = report["by_domain"].get("consumer", {}).get("top5_acc")
    otc_top5 = report["by_category"].get("OTC", {}).get("top5_acc")
    priority_top5 = report["by_tier"].get("priority", {}).get("top5_acc")
    priority_top10 = report["by_tier"].get("priority", {}).get("top10_acc")
    print("consumer-domain top5:", consumer_top5, "| OTC-category top5:", otc_top5)
    print("PRIORITY TIER (top-500 RX/OTC — the actual doctor-testing goal): "
          f"top5={priority_top5} top10={priority_top10}")
    if consumer_top5 is not None and consumer_top5 >= MIN_TOP5_CONSUMER:
        (EXPORT_DIR / "APPROVED_FOR_DEPLOY").write_text(f"consumer_top5={consumer_top5} otc_top5={otc_top5}")
        print("GATE PASSED — artifacts in", EXPORT_DIR, "are approved to copy into pill-scanner/backend.")
    else:
        print("GATE FAILED — do not deploy this model. Get more consumer-domain data or train longer.")


## Summary of what's real vs. what needs verification before trusting this run

**Verified real** (via web search at authoring time, not guessed):
Kaggle dataset slugs in Section 1.1, the ePillID/C3PI class-count and
reference/consumer image-count figures quoted in the title cell, RxNav's API
shape (already used successfully in `pill-id`'s own metadata script).

**Needs your confirmation before this notebook fully works:**
- The exact C3PI direct-download URLs (Section 1.2) — the landing page is
  real, the direct file link wasn't confirmed by fetching it.
- Every dataset's actual internal folder/filename layout (Section 1's listing
  output) — Section 3's parsing functions are a reasonable default, not a
  guarantee, for datasets I couldn't browse directly.
- All hyperparameters in Section 7 are carried over from `pill-id`'s
  `config.json` (which did produce good *offline* numbers) — they're a
  sensible starting point, not something proven optimal for this larger,
  more diverse dataset mix.
